## 0. Importações

Parte inicial, configuração de ambiente, importações e ajustes gerais. Principais ferramentas utilizdas:
- Python: Linguagem utilizada
- Pandas: Biblioteca de análise de dados
- Numpy: Dependência para a biblioteca pandas e utilitários para processo
- Json: Biblioteca para lindar com campos em formatação json

In [1]:
import pandas as pd
import numpy as np
import json
import ast
import re

from pathlib import Path

In [2]:
pip install pyarrow fastparquet

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade pandas pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
RAW_PATH       = Path('../../data/raw/ArquivosConcatenados_1.csv')
PROCESSED_PATH = Path('../../data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

## 1. Visualização inicial

Objetivos:
- Identificar natureza dos dados
- Estabelecer uma descrição precisa dos dados
- Identificar inconsistências (se for o caso)
- Realizar limpeza (se for o caso)
- Preparar dados para etapadas posteriores 

### 1.1 Importação do dataset e visualização inicial

In [5]:
# Importação do dataset
df = pd.read_csv(
    RAW_PATH,
    # parse_dates=['data_protocolo'],
    dayfirst=True,
    low_memory=True
    )

# Visualização básica
df

,incidente,classe,nome_processo,classe_extenso,tipo_processo,liminar,origem,relator,autor1,len(partes_total),...,data_protocolo,origem_orgao,lista_assuntos,len(andamentos_lista),andamentos_lista,len(decisões),decisões,len(deslocamentos),deslocamentos_lista,status_processo
0,1480010,ADI,ADI 1,AÇÃO DIRETA DE INCONSTITUCIONALIDADE,Físico,['MEDIDA LIMINAR'],RO,CÉLIO BORJA,GOVERNADOR DO ESTADO DE RONDÔNIA,4,...,06/10/1988,FÓRUM DA COMARCA DE RANCHARIA,['ASSUNTO PARA PROCESSO ANTIGO | | PROCESSO ...,21,"[{""index"": 21, ""data"": ""13/12/2023"", ""nome"": ""...",0,[],13,"[{""index"": 13, ""data_recebido"": ""Recebido em 3...",Finalizado
1,1480183,ADI,ADI 2,AÇÃO DIRETA DE INCONSTITUCIONALIDADE,Físico,['MEDIDA LIMINAR'],DF,PAULO BROSSARD,FEDERACAO NACIONAL DOS ESTABELECIMENTOS DE ENS...,3,...,12/10/1988,FÓRUM DA COMARCA DE RANCHARIA,['ASSUNTO PARA PROCESSO ANTIGO | | PROCESSO ...,28,"[{""index"": 28, ""data"": ""23/01/2007"", ""nome"": ""...",2,"[{""index"": 23, ""data"": ""06/02/1992"", ""nome"": ""...",20,"[{""index"": 20, ""data_recebido"": ""Recebido em 3...",Finalizado
2,1480234,ADI,ADI 3,AÇÃO DIRETA DE INCONSTITUCIONALIDADE,Físico,[],DF,MOREIRA ALVES,CONSELHO FEDERAL DA ORDEM DOS ADVOGADOS DO BRASIL,2,...,12/10/1988,FÓRUM DA COMARCA DE RANCHARIA,['ASSUNTO PARA PROCESSO ANTIGO | | PROCESSO ...,21,"[{""index"": 21, ""data"": ""02/10/1992"", ""nome"": ""...",2,"[{""index"": 16, ""data"": ""07/02/1992"", ""nome"": ""...",13,"[{""index"": 13, ""data_recebido"": ""NA"", ""enviado...",Finalizado
3,1480210,ADI,ADI 4,AÇÃO DIRETA DE INCONSTITUCIONALIDADE,Físico,['MEDIDA LIMINAR'],DF,SYDNEY SANCHES,PARTIDO DEMOCRÁTICO TRABALHISTA - PDT,3,...,12/10/1988,FÓRUM DA COMARCA DE RANCHARIA,['ASSUNTO PARA PROCESSO ANTIGO | | PROCESSO ...,34,"[{""index"": 34, ""data"": ""19/08/1993"", ""nome"": ""...",2,"[{""index"": 25, ""data"": ""07/03/1991"", ""nome"": ""...",13,"[{""index"": 13, ""data_recebido"": ""NA"", ""enviado...",Finalizado
4,1480556,ADI,ADI 5,AÇÃO DIRETA DE INCONSTITUCIONALIDADE,Físico,['MEDIDA LIMINAR'],SP,NÉRI DA SILVEIRA,GOVERNADOR DO ESTADO DE SÃO PAULO,2,...,18/10/1988,FÓRUM DA COMARCA DE RANCHARIA,['ASSUNTO PARA PROCESSO ANTIGO | | PROCESSO ...,19,"[{""index"": 19, ""data"": ""18/05/1992"", ""nome"": ""...",1,"[{""index"": 14, ""data"": ""27/03/1992"", ""nome"": ""...",14,"[{""index"": 14, ""data_recebido"": ""NA"", ""enviado...",Finalizado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6615,7116426,ADO,ADO 89,AÇÃO DIRETA DE INCONSTITUCIONALIDADE POR OMISSÃO,Eletrônico,['MEDIDA LIMINAR'],MG,GILMAR MENDES,ASSOCIACAO DOS POLICIAIS PENAIS DO BRASIL AGEP...,5,...,04/12/2024,SUPREMO TRIBUNAL FEDERAL,['DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE ...,8,"[{""index"": 8, ""data"": ""06/12/2024"", ""nome"": ""B...",1,"[{""index"": 5, ""data"": ""05/12/2024"", ""nome"": ""E...",4,"[{""index"": 4, ""data_recebido"": ""Recebido em 06...",Finalizado
6616,7117654,ADO,ADO 91,AÇÃO DIRETA DE INCONSTITUCIONALIDADE POR OMISSÃO,Eletrônico,['MEDIDA LIMINAR'],PA,NUNES MARQUES,ASSOCIACAO DOS POLICIAIS PENAIS DO BRASIL AGEP...,5,...,05/12/2024,SUPREMO TRIBUNAL FEDERAL,['DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE ...,18,"[{""index"": 18, ""data"": ""14/10/2025"", ""nome"": ""...",1,"[{""index"": 5, ""data"": ""19/12/2024"", ""nome"": ""D...",10,"[{""index"": 10, ""data_recebido"": ""Recebido em 1...",Em andamento
6617,7223317,ADO,ADO 92,AÇÃO DIRETA DE INCONSTITUCIONALIDADE POR OMISSÃO,Eletrônico,"['MEDIDA LIMINAR', 'TUTELA PROVISÓRIA']",DF,CÁRMEN LÚCIA,ARTICULAÇÃO DOS POVOS INDÍGENAS - APIB,13,...,11/04/2025,SUPREMO TRIBUNAL FEDERAL,['DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE ...,69,"[{""index"": 69, ""data"": ""04/02/2026"", ""nome"": ""...",1,"[{""index"": 11, ""data"": ""05/05/2025"", ""nome"": ""...",10,"[{""index"": 10, ""data_recebido"": ""Recebido em 1...",Em andamento
6618,7272296,ADO,ADO 93,AÇÃO DIRETA DE INCONSTITUCIONALIDADE POR OMISSÃO,Eletrônico,['MEDIDA LIMINAR'],DF,CÁRMEN LÚCIA,INSTITUTO NACIONAL DE

In [6]:
# Colunas do dataset
df.columns.unique()

Index(['incidente', 'classe', 'nome_processo', 'classe_extenso',
       'tipo_processo', 'liminar', 'origem', 'relator', 'autor1',
       'len(partes_total)', 'partes_total', 'data_protocolo', 'origem_orgao',
       'lista_assuntos', 'len(andamentos_lista)', 'andamentos_lista',
       'len(decisões)', 'decisões', 'len(deslocamentos)',
       'deslocamentos_lista', 'status_processo'],
      dtype='str')

#### Descrição das variáveis do conjunto de dados

A tabela abaixo descreve cada uma das 21 colunas presentes no dataset, que contém informações processuais de ações judiciais (principalmente ADI e ADO) do Supremo Tribunal Federal (STF).

| Variável | Descrição |
|----------|-----------|
| `incidente` | Número de identificação do processo no sistema (aparentemente um código numérico único). |
| `classe` | Sigla da classe processual (ex.: ADI, ADO). |
| `nome_processo` | Nome resumido do processo, geralmente composto pela classe seguida de um número sequencial. |
| `classe_extenso` | Nome completo da classe processual (ex.: "AÇÃO DIRETA DE INCONSTITUCIONALIDADE"). |
| `tipo_processo` | Indica se o processo tramita em meio físico ou eletrônico. |
| `liminar` | Lista com os tipos de tutela de urgência requeridos (ex.: "MEDIDA LIMINAR", "TUTELA PROVISÓRIA"). Quando vazio (`[]`), indica ausência de pedido liminar. |
| `origem` | Sigla da unidade da federação de origem do processo (ex.: RO, DF, SP, MG, PA). |
| `relator` | Nome do ministro relator do processo no STF. |
| `autor1` | Nome da primeira parte autora (pessoa física, jurídica, órgão público ou entidade). |
| `len(partes_total)` | Número total de partes envolvidas no processo (autores, réus, interessados, etc.). |
| `data_protocolo` | Data de protocolo (entrada) do processo no tribunal, no formato `dd/mm/aaaa`. |
| `origem_orgao` | Órgão ou unidade judiciária de onde se originou o processo (ex.: "FÓRUM DA COMARCA DE RANCHARIA", "SUPREMO TRIBUNAL FEDERAL"). |
| `lista_assuntos` | Lista de assuntos/cadastros temáticos atribuídos ao processo (ex.: "DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO"). |
| `len(andamentos_lista)` | Número de eventos (andamentos) registrados na movimentação do processo. |
| `andamentos_lista` | Lista de dicionários contendo detalhes dos andamentos, como índice, data, nome do ato, descrição, etc. |
| `len(decisões)` | Quantidade de decisões judiciais proferidas no processo. |
| `decisões` | Lista de dicionários com informações das decisões, incluindo índice, data, nome da decisão, tipo e conteúdo. |
| `len(deslocamentos)` | Número de deslocamentos (movimentações físicas ou eletrônicas) do processo entre unidades ou órgãos. |
| `deslocamentos_lista` | Lista de dicionários com dados dos deslocamentos, como data de recebimento, data de envio, origem, destino, etc. |
| `status_processo` | Situação atual do processo: "Finalizado" ou "Em andamento". |

> **Observação:** As variáveis com prefixo `len()` representam a contagem de elementos das respectivas listas (andamentos, decisões, deslocamentos). As listas estão estruturadas no formato JSON, conforme exemplos na tabela HTML.

### 1.2 Análise descritiva do dataset

In [7]:
# Obtendo uma análise descritiva do dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6620 entries, 0 to 6619
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   incidente              6620 non-null   int64
 1   classe                 6620 non-null   str  
 2   nome_processo          6620 non-null   str  
 3   classe_extenso         6620 non-null   str  
 4   tipo_processo          6620 non-null   str  
 5   liminar                6620 non-null   str  
 6   origem                 6602 non-null   str  
 7   relator                6570 non-null   str  
 8   autor1                 6613 non-null   str  
 9   len(partes_total)      6620 non-null   int64
 10  partes_total           6620 non-null   str  
 11  data_protocolo         6620 non-null   str  
 12  origem_orgao           6443 non-null   str  
 13  lista_assuntos         6620 non-null   str  
 14  len(andamentos_lista)  6620 non-null   int64
 15  andamentos_lista       6620 non-null   str  
 16 

#### Dimensões
- **Total de linhas:** 6.620
- **Total de colunas:** 21

#### Tipos de dados por coluna

| Tipo   | Quantidade | Colunas                                                                 |
|--------|------------|-------------------------------------------------------------------------|
| `int64`| 5          | `incidente`, `len(partes_total)`, `len(andamentos_lista)`, `len(decisões)`, `len(deslocamentos)` |
| `str`  | 16         | Todas as demais (incluindo campos que armazenam listas em formato JSON) |

#### Valores ausentes (non-null vs total)

| Coluna               | Não-nulos | Nulos | % Nulos | Observação                                     |
|----------------------|-----------|-------|---------|------------------------------------------------|
| `origem`             | 6.602     | 18    | 0,27%   | Poucos registros sem informação de UF          |
| `relator`            | 6.570     | 50    | 0,76%   | Pequena proporção sem relator identificado     |
| `autor1`             | 6.613     | 7     | 0,11%   | Raríssimos casos sem autor principal           |
| `origem_orgao`       | 6.443     | 177   | 2,67%   | Maior lacuna; órgão de origem não informado    |
| **Demais colunas**   | 6.620     | 0     | 0%      | Nenhum valor faltante                          |

> A coluna `origem_orgao` é a que apresenta maior número de ausências (177 linhas), possivelmente por processos muito antigos ou com cadastro incompleto.

#### Detalhamento de colunas com conteúdo semi-estruturado

Várias colunas armazenam **listas em formato string** (JSON-like). Exemplos:

- `liminar`: ex. `"['MEDIDA LIMINAR']"` ou `"[]"`
- `partes_total`: lista de todas as partes do processo
- `lista_assuntos`: lista de assuntos temáticos
- `andamentos_lista`: lista de dicionários com movimentações
- `decisões`: lista de dicionários com decisões judiciais
- `deslocamentos_lista`: lista de dicionários com registro de envios/recebimentos

**Implicação:** Para análises mais aprofundadas, será necessário converter essas strings em estruturas de lista/dicionário usando `ast.literal_eval()` ou `json.loads()`.

#### Estatísticas preliminares (exemplo das primeiras linhas)

- **`incidente`**: números que variam de ~1.480.000 a ~7.280.000, aparentemente sequenciais por ano.
- **`classe`**: predominância de `ADI` (Ação Direta de Inconstitucionalidade) e `ADO` (Ação Direta de Inconstitucionalidade por Omissão).
- **`tipo_processo`**: `Físico` (processos antigos, anteriores a meados da década de 2010) e `Eletrônico` (processos mais recentes).
- **`data_protocolo`**: abrangência desde 06/10/1988 até 05/06/2025 (datas futuras indicam processos em andamento com movimentação prevista).
- **`status_processo`**: duas categorias – `Finalizado` e `Em andamento`.

#### Potenciais problemas ou pontos de atenção

1. **Datas no formato `dd/mm/aaaa`** – não estão como tipo `datetime`, exigindo conversão.
2. **Listas como strings** – inviabiliza contagens diretas (por isso existem colunas auxiliares `len(...)` já calculadas).
3. **Processos com `data_protocolo` posterior à data atual** (ex.: 14/10/2025, 04/02/2026) – podem ser datas previstas ou erros de digitação; requer verificação.
4. **`origem_orgao`** com valores como `"FÓRUM DA COMARCA DE RANCHARIA"` – provavelmente um valor genérico ou placeholder para processos antigos migrados.

### A título de curiosidade: Tabela cruzada das classes e tipos

In [8]:
tabela_cruzada = pd.crosstab(df['classe'], df['tipo_processo'], margins=True, margins_name='Total')
print(tabela_cruzada)

tipo_processo  Eletrônico  Físico  Total
classe                                  
ADC                    79      21    100
ADI                  2311    3546   5857
ADO                    78       5     83
ADPF                  580       0    580
Total                3048    3572   6620


- ADI é a classe predominante (5.857 processos, ~88,5% do total).
- ADPF aparece apenas como eletrônico (580 processos).
- ADO (Ação Direta de Inconstitucionalidade por Omissão) é a classe menos frequente (83 processos), com apenas 5 físicos.
- ADC (Ação Declaratória de Constitucionalidade) tem 100 processos, com 79 eletrônicos e 21 físicos.
- Processos físicos ainda são maioria (3.572 vs 3.048 eletrônicos), mas isso pode refletir processos muito antigos. Se você cruzar com data_protocolo, verá que físicos são quase todos anteriores a ~2014.

### 1.3 Checagem JSONs 

In [9]:
# Checar um JSON de andamentos pra ver a estrutura interna
sample = df['andamentos_lista'].dropna().iloc[0]
print(json.loads(sample))

[{'index': 21, 'data': '13/12/2023', 'nome': 'Petição', 'complemento': 'Amicus curiae - Petição: 139250 Data: 13/12/2023, às 18:07:32', 'julgador': 'NA', 'validade': 'invalid', 'link': 'NA', 'link_tipo': 'NA', 'link_conteúdo': 'NA'}, {'index': 20, 'data': '11/09/2013', 'nome': 'Petição', 'complemento': 'Juntada de documentos - Petição: 45190 Data: 11/09/2013 19:59:29.657 GMT-03:00', 'julgador': 'NA', 'validade': 'valid', 'link': 'NA', 'link_tipo': 'NA', 'link_conteúdo': 'NA'}, {'index': 19, 'data': '11/09/2013', 'nome': 'Petição', 'complemento': 'Juntada de documentos - Petição: 45174 Data: 11/09/2013 18:45:42.65 GMT-03:00', 'julgador': 'NA', 'validade': 'valid', 'link': 'NA', 'link_tipo': 'NA', 'link_conteúdo': 'NA'}, {'index': 18, 'data': '10/03/1992', 'nome': 'BAIXA AO ARQUIVO DO STF', 'complemento': '', 'julgador': 'NA', 'validade': 'valid', 'link': 'NA', 'link_tipo': 'NA', 'link_conteúdo': 'NA'}, {'index': 17, 'data': '10/03/1992', 'nome': 'DECORRIDO O PRAZO', 'complemento': 'INTE

A estrutura é consistente e bem definida, cada andamento tem sempre os mesmos campos: index, data, nome, complemento, julgador, validade, link, link_tipo, link_conteúdo. Isso é bom. Mas tem alguns problemas que vão aparecer na análise:
- 'NA' como string em vez de None/NaN real — vai quebrar filtros
- validade com valor 'invalid' em alguns registros — precisa entender o que significa
- data como string dentro do JSON, não datetime
- julgador quase sempre 'NA'
- complemento com texto livre e inconsistente — difícil de usar diretamente

O caminho mais robusto e alinhado com é explodir o JSON em um DataFrame separado e salvar como parquet. 

### 1.4 Chacagem de HTML sujo em lista_assuntos

In [10]:
# Ver o HTML sujo em lista_assuntos
print(df['lista_assuntos'].iloc[0])

['ASSUNTO PARA PROCESSO ANTIGO |   | PROCESSO ANTIGO']


Isso é uma string que parece lista com um único valor de texto usando | como separador interno. A limpeza é trivial.

## 2. Limpeza

#### 2.1 Limpeza da tabela principal de processos 

In [11]:
# Cria uma cópia do dataset original por precaução
proc = df.copy()

Criando uma cópia do dataset original

#### 2.1.1 Ajuste de tipos básicos

In [12]:
proc['incidente'] = proc['incidente'].astype(int)
proc['data_protocolo'] = pd.to_datetime(proc['data_protocolo'], dayfirst=True, errors='coerce')

- incidente → inteiro.
- data_protocolo → tipo datetime (dia/mês/ano). Erros de conversão viram NaT.

#### 2.1.2 Colunas categóricas (economia de memória)

In [13]:
for col in ['classe', 'classe_extenso', 'tipo_processo', 'origem', 'status_processo']:
    proc[col] = proc[col].astype('category')

Converte colunas com poucos valores únicos para o tipo category. Isso reduz o uso de memória e acelera agrupamentos.

#### 2.1.3 Substituição de strings nulas por None

In [14]:
STR_NULOS = {'NA', 'nan', 'None', '', ' '}
for col in proc.select_dtypes('object').columns:
    proc[col] = proc[col].apply(lambda x: None if (isinstance(x, str) and x.strip() in STR_NULOS) else x)

/tmp/ipykernel_815/2678883157.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in proc.select_dtypes('object').columns:


Para todas as colunas do tipo object (strings), valores como 'NA', 'nan', 'None', '', ' ' são substituídos por None (valor nulo real). Isso uniformiza os dados ausentes.

#### 2.1.4 Tratamento da coluna liminar


In [15]:
def parse_liminar(valor):
    """Converte string '["MEDIDA LIMINAR"]' em lista Python."""

    if pd.isna(valor) or valor is None:
        return []

    try:
        resultado = ast.literal_eval(valor)
        return resultado if isinstance(resultado, list) else []
    
    except Exception:
        return []
    
proc['liminar_lista'] = proc['liminar'].apply(parse_liminar)
proc['tem_liminar'] = proc['liminar_lista'].apply(lambda x: len(x) > 0)
proc['qtd_liminares'] = proc['liminar_lista'].apply(len)

- liminar_lista: lista real de liminares (ex.: ['MEDIDA LIMINAR', 'TUTELA PROVISÓRIA']).
- tem_liminar: booleano indicando se o processo possui alguma liminar.
- qtd_liminares: número de liminares (útil para processos com múltiplos pedidos).

#### 2.1.5 Tratamento da coluna lista_assuntos

In [16]:
def parse_assuntos(valor):
    """Converte string de lista de assuntos em lista de strings limpas."""

    if pd.isna(valor) or valor is None:
        return []
    
    try:
        lista = ast.literal_eval(valor)
        assuntos = []

        for item in lista:
            # remove &nbsp; e espaços extras, split no separador |
            partes = [p.strip().replace('\xa0', '').replace('&nbsp;', '') for p in item.split('|')]
            partes = [p for p in partes if p]  # remove strings vazias
            assuntos.extend(partes)

        return list(dict.fromkeys(assuntos))  # remove duplicatas mantendo ordem
    
    except Exception:
        return []

proc['assuntos_lista']    = proc['lista_assuntos'].apply(parse_assuntos)
proc['assunto_principal'] = proc['assuntos_lista'].apply(
    lambda x: x[0] if x else None
)

- Limpa entidades HTML como &nbsp; e espaços extras.
- Achata a hierarquia de assuntos (separada por |).
- Remove duplicatas mantendo ordem.
- assunto_principal: primeiro assunto da lista (útil para classificação geral).

In [17]:
proc.info()

<class 'pandas.DataFrame'>
RangeIndex: 6620 entries, 0 to 6619
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   incidente              6620 non-null   int64         
 1   classe                 6620 non-null   category      
 2   nome_processo          6620 non-null   str           
 3   classe_extenso         6620 non-null   category      
 4   tipo_processo          6620 non-null   category      
 5   liminar                6620 non-null   str           
 6   origem                 6602 non-null   category      
 7   relator                6570 non-null   str           
 8   autor1                 6613 non-null   str           
 9   len(partes_total)      6620 non-null   int64         
 10  partes_total           6620 non-null   str           
 11  data_protocolo         6620 non-null   datetime64[us]
 12  origem_orgao           6443 non-null   str           
 13  lista_assuntos

In [18]:
print('Nulos por coluna (colunas escalares):')
colunas_escalares = proc.drop(columns=[
    'andamentos_lista', 'decisões', 'deslocamentos_lista',
    'partes_total', 'liminar_lista', 'assuntos_lista'
]).isnull().sum()
print(colunas_escalares[colunas_escalares > 0])

Nulos por coluna (colunas escalares):
origem                18
relator               50
autor1                 7
origem_orgao         177
assunto_principal    369
dtype: int64


É possível ver 369 valores nulos em assunto principal após o processamento.

In [19]:
proc[proc['assunto_principal'].isna()][['incidente', 'lista_assuntos']].head()

,incidente,lista_assuntos
199,1494727,[]
1511,1652307,[]
1684,1684757,[]
1685,1684932,[]
1686,1685085,[]


Listando algumas linhas com lista assuntos null.

In [20]:
# Amostra dos incidentes problemáticos
incidentes_problema = [1494727, 1652307, 1684757, 1684932, 1685085]

# Exibe lista_assuntos original
print(df[df['incidente'].isin(incidentes_problema)][['incidente', 'lista_assuntos']])

      incidente lista_assuntos
199     1494727             []
1511    1652307             []
1684    1684757             []
1685    1684932             []
1686    1685085             []


Ao observar os mesmos lista_assunto no dataset original, observamos que os mesmos são null. Portanto, os 369 valores nulos são devido ao dataset.

### 3. Explosão dos JSONs aninhados

In [21]:
def explodir_json(
    df: pd.DataFrame,
    coluna: str,
    prefixo: str,
    colunas_contexto: list = None
) -> pd.DataFrame:
    """
    Transforma uma coluna de JSON aninhado em DataFrame normalizado.

    Parâmetros
    ----------
    df               : DataFrame principal
    coluna           : nome da coluna com JSON string
    prefixo          : prefixo aplicado às colunas do JSON (ex: 'and_')
    colunas_contexto : colunas do df principal a incluir em cada linha
    """
    if colunas_contexto is None:
        colunas_contexto = ['incidente', 'classe', 'tipo_processo']

    registros = []
    erros = []

    for _, row in df.iterrows():
        
        try:
            itens = ast.literal_eval(str(row[coluna]))
            if not isinstance(itens, list):
                continue

            for item in itens:

                entrada = {col: row[col] for col in colunas_contexto}
                for k, v in item.items():
                    # 'NA' string → None real
                    v_clean = None if (isinstance(v, str) and v.strip() in STR_NULOS) else v

                    # string vazia → None
                    v_clean = None if v_clean == '' else v_clean
                    entrada[f'{prefixo}{k}'] = v_clean

                registros.append(entrada)

        except Exception as e:
            erros.append({'incidente': row.get('incidente'), 'erro': str(e)})

    if erros:
        print(f'[{coluna}] {len(erros)} erros de parsing:')

        for e in erros[:5]:
            print(f"  incidente {e['incidente']}: {e['erro']}")

    return pd.DataFrame(registros)

#### 3.1 Andamento

In [ ]:
df_andamentos = explodir_json(proc, 'andamentos_lista', 'and_')

# tipos
df_andamentos['and_data'] = pd.to_datetime(
    df_andamentos['and_data'], format='%d/%m/%Y', errors='coerce'
)
df_andamentos['and_index'] = pd.to_numeric(df_andamentos['and_index'], errors='coerce').astype('Int64')

# flag de andamento relevante para pesquisa: julgamento virtual
TERMOS_VIRTUAL = [
    'plenário virtual', 'pauta virtual', 'julgamento virtual',
    'incluído em pauta virtual', 'retirado de pauta virtual'
]
def is_virtual(texto):
    if pd.isna(texto):
        return False
    return any(t in str(texto).lower() for t in TERMOS_VIRTUAL)

df_andamentos['and_is_virtual'] = (
    df_andamentos['and_nome'].apply(is_virtual) |
    df_andamentos['and_complemento'].apply(is_virtual)
)

In [ ]:
print(f'Shape: {df_andamentos.shape}')
print(f'Andamentos virtuais identificados: {df_andamentos["and_is_virtual"].sum()}')
print(f'Nulos em and_data: {df_andamentos["and_data"].isna().sum()}')
df_andamentos.head(3)

chunk 1 ok
chunk 2 ok
chunk 3 ok
chunk 4 ok
chunk 5 ok
chunk 6 ok
chunk 7 ok
chunk 8 ok
chunk 9 ok
chunk 10 ok
chunk 11 ok
chunk 12 ok
chunk 13 ok
chunk 14 ok
chunk 15 ok
chunk 16 ok
chunk 17 ok
chunk 18 ok
chunk 19 ok
chunk 20 ok
chunk 21 ok
chunk 22 ok
chunk 23 ok
chunk 24 ok
chunk 25 ok
chunk 26 ok
chunk 27 ok
chunk 28 ok
chunk 29 ok
chunk 30 ok
chunk 31 ok
chunk 32 ok
chunk 33 ok
chunk 34 ok
chunk 35 ok
chunk 36 ok
chunk 37 ok
chunk 38 ok
chunk 39 ok
chunk 40 ok
chunk 41 ok
chunk 42 ok
chunk 43 ok
chunk 44 ok
chunk 45 ok
chunk 46 ok
chunk 47 ok
chunk 48 ok
chunk 49 ok
chunk 50 ok
chunk 51 ok
chunk 52 ok
chunk 53 ok
chunk 54 ok
chunk 55 ok
chunk 56 ok
chunk 57 ok
chunk 58 ok
chunk 59 ok


In [ ]:
print(f'Shape: {df_andamentos.shape}')
print(f'Andamentos virtuais identificados: {df_andamentos["and_is_virtual"].sum()}')
print(f'Nulos em and_data: {df_andamentos["and_data"].isna().sum()}')
df_andamentos.head(3)

NameError: name 'df_andamentos' is not defined

In [ ]:
df_andamentos.info()

<class 'pandas.DataFrame'>
RangeIndex: 349037 entries, 0 to 349036
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   incidente          349037 non-null  int64         
 1   classe             349037 non-null  str           
 2   tipo_processo      349037 non-null  str           
 3   and_index          349037 non-null  Int64         
 4   and_data           349037 non-null  datetime64[us]
 5   and_nome           349037 non-null  str           
 6   and_complemento    280203 non-null  str           
 7   and_julgador       17662 non-null   str           
 8   and_validade       349037 non-null  str           
 9   and_link           29309 non-null   str           
 10  and_link_tipo      0 non-null       object        
 11  and_link_conteúdo  29261 non-null   str           
 12  and_is_virtual     349037 non-null  bool          
dtypes: Int64(1), bool(1), datetime64[us](1), int64(1), obje

#### 3.2 Decisões

In [ ]:
df_decisoes = explodir_json(proc, 'decisões', 'dec_')

df_decisoes['dec_data'] = pd.to_datetime(
    df_decisoes['dec_data'], format='%d/%m/%Y', errors='coerce'
)

df_decisoes['dec_index'] = pd.to_numeric(df_decisoes['dec_index'], errors='coerce').astype('Int64')

Shape: (17662, 12)


,incidente,classe,tipo_processo,dec_index,dec_data,dec_nome,dec_complemento,dec_julgador,dec_validade,dec_link,dec_link_tipo,dec_link_conteúdo
0,1480183,ADI,Físico,23,1992-02-06,JULGAMENTO DO PLENO - NAO CONHECIDO,"POR MAIORIA DE VOTOS, O TRIBUNAL NÃO CONHECEU ...",TRIBUNAL PLENO,valid,NaN,None,NaN
1,1480183,ADI,Físico,3,1988-10-20,JULGAMENTO NO PLENO,INDEF UNAN DJ 26/10/88,TRIBUNAL PLENO,valid,NaN,None,NaN
2,1480234,ADI,Físico,16,1992-02-07,JULGAMENTO DO PLENO - NAO CONHECIDO,"POR VOTAÇÃO UNÂNIME, O TRIBUNAL NÃO CONHECEU D...",TRIBUNAL PLENO,valid,NaN,None,NaN


In [ ]:
print(f'Shape: {df_decisoes.shape}')
df_decisoes.head(3)

In [ ]:
df_decisoes.info()

#### 3.2 Deslocamento

In [ ]:
df_deslocamentos = explodir_json(proc, 'deslocamentos_lista', 'desl_')

# datas de deslocamento aparecem em texto livre: "Recebido em 30/01/2024"
def extrair_data_desl(texto):
    """Extrai data no formato DD/MM/AAAA de strings de deslocamento."""

    if pd.isna(texto):
        return None
    
    match = re.search(r'(\d{2}/\d{2}/\d{4})', str(texto))
    return match.group(1) if match else None

for col_data in ['desl_data_recebido', 'desl_enviado ']:
    
    if col_data in df_deslocamentos.columns:
        df_deslocamentos[col_data + '_dt'] = pd.to_datetime(
            df_deslocamentos[col_data].apply(extrair_data_desl),
            format='%d/%m/%Y', errors='coerce'
        )

Shape: (199801, 9)


,incidente,classe,tipo_processo,desl_index,desl_data_recebido,desl_enviado por,desl_recebido por,desl_guia,desl_data_recebido_dt
0,1480010,ADI,Físico,13,Recebido em 30/01/2021,"COORDENADORIA DE GESTÃO DA INFORMAÇÃO, MEMÓRIA...",Enviado por COORDENADORIA DE MEMÓRIA E GESTÃO ...,Guia 6/2021,2021-01-30
1,1480010,ADI,Físico,12,Recebido em 13/06/2019,COORDENADORIA DE MEMÓRIA E GESTÃO DOCUMENTAL,Enviado por SEÇÃO DE ARQUIVO em 13/06/2019,Guia 385/2019,2019-06-13
2,1480010,ADI,Físico,11,Recebido em 11/11/2014,SEÇÃO DE ARQUIVO,Enviado por SEÇÃO DE ARQUIVO em 11/11/2014,Guia 309/2014,2014-11-11


In [ ]:
print(f'Shape: {df_deslocamentos.shape}')
df_deslocamentos.head(3)

In [ ]:
df_deslocamentos.info()

### 4. Tabela final e validação

#### 4.1 Tabela final

In [ ]:
COLUNAS_REMOVER = [
    'andamentos_lista',
    'decisões',
    'deslocamentos_lista',
    'liminar',          # substituída por liminar_lista + tem_liminar
    'lista_assuntos',   # substituída por assuntos_lista + assunto_principal
]

processos_final = proc.drop(columns=[c for c in COLUNAS_REMOVER if c in proc.columns])

print('Colunas finais:')
print(processos_final.dtypes)
print(f'\nShape final: {processos_final.shape}')

Colunas finais:
incidente                         int64
classe                         category
nome_processo                       str
classe_extenso                 category
tipo_processo                  category
origem                         category
relator                             str
autor1                              str
len(partes_total)                 int64
partes_total                        str
data_protocolo           datetime64[us]
origem_orgao                        str
len(andamentos_lista)             int64
len(decisões)                     int64
len(deslocamentos)                int64
status_processo                category
assuntos_lista                   object
assunto_principal                   str
liminar_lista                    object
tem_liminar                        bool
qtd_liminares                     int64
dtype: object

Shape final: (6620, 21)


Remove colunas JSON brutas (já exportadas separadamente) e colunas originais
que foram substituídas por versões limpas.

#### 4.2 Validação

In [ ]:
print('=== VALIDAÇÕES ===')

# 1. Sem duplicatas de incidente
dupl = processos_final['incidente'].duplicated().sum()
print(f'Incidentes duplicados: {dupl}  {"✓" if dupl == 0 else "⚠️ VERIFICAR"}')

# 2. Datas dentro do esperado (1988 – hoje)
fora_range = processos_final[
    (processos_final['data_protocolo'].dt.year < 1988) |
    (processos_final['data_protocolo'].dt.year > 2026)
]
print(f'Datas fora do range 1988–2026: {len(fora_range)}  {"✓" if len(fora_range) == 0 else "⚠️ VERIFICAR"}')

# 3. Classes esperadas
classes_ok = set(processos_final['classe'].unique()) <= {'ADI', 'ADC', 'ADO', 'ADPF'}
print(f'Apenas classes ADI/ADC/ADO/ADPF: {"✓" if classes_ok else "⚠️ VERIFICAR"}')
print(f'Classes encontradas: {processos_final["classe"].unique().tolist()}')

# 4. tipo_processo só Físico/Eletrônico
tipos_ok = set(processos_final['tipo_processo'].unique()) <= {'Físico', 'Eletrônico'}
print(f'Apenas Físico/Eletrônico: {"✓" if tipos_ok else "⚠️ VERIFICAR"}')

# 5. Integridade dos JSONs explodidos
print(f'\nTotal andamentos extraídos: {len(df_andamentos)}')
print(f'Total decisões extraídas:   {len(df_decisoes)}')
print(f'Total deslocamentos:        {len(df_deslocamentos)}')

=== VALIDAÇÕES ===
Incidentes duplicados: 0  ✓
Datas fora do range 1988–2026: 0  ✓
Apenas classes ADI/ADC/ADO/ADPF: ✓
Classes encontradas: ['ADI', 'ADC', 'ADPF', 'ADO']
Apenas Físico/Eletrônico: ✓

Total andamentos extraídos: 349037
Total decisões extraídas:   17662
Total deslocamentos:        199801


### 5. Exportação para `parquet`

In [ ]:
# Parquet não suporta colunas com listas nativas — converter para string JSON
for col in ['liminar_lista', 'assuntos_lista', 'partes_total']:
    if col in processos_final.columns:
        processos_final[col] = processos_final[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
        )

# Exportar
processos_final.to_parquet(PROCESSED_PATH / 'processos.parquet',    index=False)
df_andamentos.to_parquet(  PROCESSED_PATH / 'andamentos.parquet',   index=False)
df_decisoes.to_parquet(    PROCESSED_PATH / 'decisoes.parquet',     index=False)
df_deslocamentos.to_parquet(PROCESSED_PATH / 'deslocamentos.parquet', index=False)

print('Exportação concluída:')
for f in PROCESSED_PATH.glob('*.parquet'):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name}: {size_mb:.2f} MB')

ArrowKeyError: A type extension with name pandas.period already defined